In [ ]:
%run ./initialize.ipynb

In [ ]:
import lion_tools
import ipynbname
spark=lion_tools.get_or_create_spark()
lion_tools.extend_dataframe()
spark.createDataFrame([[1, "Alice]\\n in hell{}"], [2, "Bob"]], ["id", "name"]).eDisplay(
    file_path=str(ipynbname.path().parent.parent.joinpath("output", "unknown_dataframe_view.html"))
)

In [ ]:
import pyspark.sql.functions as F

data = [
    [i, i * 2, f'bla_bla_{i % 5}', f'string_{i % 4}']
    for i in range(1001)
]
data[2][2] = None

df = (
    spark.createDataFrame(data, ['id', 'some_num', 'some_str', 'another_str'])
    .selectExpr(
        '*',
        'if(random() <0.05, null, cast(1000*random() as int)) as random_int',
        'round(10*random(), 2) as random_double',
    )
)

base = (
    df
    .filter("not(id = 5 and some_num = 10)")
    .drop('random_int')
)
compare = (
    # df with some changes we will test to see if compare_summary will detect them
    df
    .withColumn('some_num', F.when(F.col('id') == 5, 999).otherwise(F.col('some_num')))
)

(
    base
    .eCompareSummary(
        compare, 
        'some_str',
        stats=[
            "min", "max", "sum", "avg", "avg_null", "count_null",  "count_not_null",
            "count_distinct", "approx_count_distinct"
        ],
        ignore_missing_columns=False,
        round_decimals=2,
    )
    # .orderBy(F.col('column_no__base').asc(), F.col('column_no__compare').asc())
    # .eC(True, name=f'compare stats', lazy=False, add_time_to_name=True)
    # .select("column", "result", "column_no__base", "column_no__compare")
    # .show(truncate=False)
)

